# National Ground-Water Monitoring Network (NGWMN)

The [National Ground-Water Monitoring Network](https://cida.usgs.gov/ngwmn/) (NGWMN)
brings groundwater data from many state, federal, and local agencies into a single
location. USGS exposes it through a dedicated OGC API
(`https://api.waterdata.usgs.gov/ngwmn/ogcapi`), which `dataretrieval` wraps in the
`dataretrieval.ngwmn` module — a sibling of `dataretrieval.waterdata` built on the
same shared OGC engine, so chunking, pagination, and result shaping behave the same.

There are five getters:

| Function | Description |
| --- | --- |
| `get_sites` | Monitoring-location (well) metadata |
| `get_water_level` | Water-level observations |
| `get_lithology` | Lithology (geologic material) logs |
| `get_well_construction` | Well-construction records |
| `get_providers` | Contributing data providers |

Unlike the main Water Data collections, NGWMN aggregates locations from many
agencies, so `monitoring_location_id` values use agency prefixes besides `USGS-`
(e.g. `MBMG-702934`, `AKDNR-535134236016630`).

In [ ]:
from dataretrieval import ngwmn

## Providers

List the organizations contributing data, optionally filtered by state.

In [ ]:
providers, md = ngwmn.get_providers(state="WI")
print(f"{len(providers)} providers in WI")
providers.head()

## Sites

`get_sites` returns well metadata. Sites carry geometry by default, so the result is a
`GeoDataFrame`; pass `skip_geometry=True` to drop it.

In [ ]:
sites, md = ngwmn.get_sites(state="Wisconsin")
print(f"{len(sites)} NGWMN sites in Wisconsin")
sites[["monitoring_location_id", "monitoring_location_name", "national_aquifer_description"]].head()

## Water levels

`get_water_level` returns the observations for one or more sites. A two-element
`datetime=[start, end]` restricts the record to a time window; a list of
`monitoring_location_id`s fans out across sites and is unioned.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

site = "USGS-272838082142201"
wl, md = ngwmn.get_water_level(monitoring_location_id=site)
print(f"{len(wl)} water-level observations at {site}")

wl["sample_time"] = pd.to_datetime(wl["sample_time"], errors="coerce", utc=True)
wl = wl.dropna(subset=["sample_time"]).sort_values("sample_time")
depth = pd.to_numeric(wl["water_depth_below_land_surface_ft"], errors="coerce")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(wl["sample_time"], depth, lw=0.8)
ax.invert_yaxis()  # depth increases downward
ax.set(xlabel="Date", ylabel="Depth to water (ft below land surface)",
       title=f"NGWMN water levels \u2014 {site}")
plt.tight_layout()
plt.show()

Restrict to a date range, or query several sites at once (they fan out and
union):

In [ ]:
windowed, md = ngwmn.get_water_level(
    monitoring_location_id=site, datetime=["2022-01-01", "2024-01-01"]
)
print(f"{len(windowed)} observations in 2022\u20132024")

multi, md = ngwmn.get_water_level(
    monitoring_location_id=["USGS-272838082142201", "USGS-404159100494601"]
)
print(f"{multi['monitoring_location_id'].nunique()} sites, {len(multi)} observations")

## Well construction and lithology

Construction records describe a well's physical build-out; lithology logs describe the
geologic materials with depth.

In [ ]:
construction, md = ngwmn.get_well_construction(monitoring_location_id=site)
construction[["monitoring_location_obs_number", "type", "material", "depth_from", "depth_to"]].head()

In [ ]:
lithology, md = ngwmn.get_lithology(monitoring_location_id="AKDNR-535134236016630")
lithology[["lithology_depth_from", "lithology_depth_to", "lithology_description"]].head()